In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
Working_directory = 'Loop_1x1'  #@param {type:"string"}

work_dir = '/content/drive/MyDrive/AML/AML_Project/'+Working_directory

!mkdir -p $work_dir
cur_work_dir = work_dir

print("Current working directory on google drive: ")
os.chdir(work_dir)

!pwd

Current working directory on google drive: 
/content/drive/MyDrive/AML/AML_Project/Loop_1x1


In [ ]:
input_file = "good_clips/detailedbp.txt"
output_file = "good_clips/detailedbp.csv"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        new_line = line.replace("#", ",")
        outfile.write(new_line)

In [ ]:
input_file = "good_clips/bp.txt"
output_file = "good_clips/bp.csv"

with open(input_file, "r") as infile, open(output_file, "w") as outfile:
    for line in infile:
        new_line = line.replace("#", ",")
        outfile.write(new_line)

In [ ]:
#Set up
import os
import csv
import numpy as np

input_folder = work_dir+'/good/good_clips/'
output_file = Working_directory+"_all_bases.csv"

seq_len = 10 #@param {type:"integer"}

rows = []

for filename in os.listdir(input_folder):
    if filename.endswith(".PDB"):
        name = filename[:-4]
        parts = name.split("_")

        # Need at least id1 + sequence + something else
        if len(parts) < 3:
            print(f"Skipping malformed filename: {filename}")
            continue

        id1 = parts[0]
        sequence = parts[1]
        rest_id = "_".join(parts[2:])  # handles id2, id2_id3, etc.

        combined_id = f"{id1}_{rest_id}"

        # Validate sequence
        if len(sequence) != seq_len:
            print(f"Skipping (sequence not correct length): {filename}")
            continue
        # Mark motif bases
        motif_baseno = {3,8}

        for i, base in enumerate(sequence, start=1):
          is_motif = i in motif_baseno
          rows.append([combined_id, i, base, is_motif])



# Write CSV
with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "position", "base", "is_motif"])
    writer.writerows(rows)

print(f"CSV written to {output_file}")

CSV written to Loop_1x1_all_bases.csv


In [ ]:
#Stacking
input_file=work_dir+"/good/good_clips/stacks.csv"
output_file="all_stacks.csv"



with open(input_file, "r") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ["id", "base", "position", "<>", "<<", ">>"]
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in reader:
        value = row["sequence"]
        nt1 = int(row["nt1"])
        nt2 = int(row["nt2"])
        stacking = row["stacking"]

        parts = value.split("_")
        if len(parts) < 3:
            continue

        id_part = parts[0]
        sequence = parts[1]
        rest = "_".join(parts[2:])
        id_rest = f"{id_part}_{rest}"

        for i, base in enumerate(sequence, start=1):
            out_row = {
                "id": id_rest,
                "base": base,
                "position": i,
                "<>": 0,
                "<<": 0,
                ">>": 0
            }

            if stacking in out_row:
                if i == nt1:
                    out_row[stacking] = 1
                elif i == nt2:
                    out_row[stacking] = 2

            writer.writerow(out_row)



In [ ]:
#detailed bp

input_file=work_dir+"/good/good_clips/detailedbp.csv"
output_file="all_detailedbp.csv"

columns = ["hbond1", "hbond2", "hbond3", "hbond4", "hbond5"]

unique_values = set()

with open(input_file, "r") as infile:
    reader = csv.DictReader(infile)

    for row in reader:
        for col in columns:
            value = (row[col] or "").strip()
            if value:  # skip empty cells
                unique_values.add(value)

# convert to list
unique_list = sorted(unique_values)
print(unique_list)
with open(input_file, "r") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ["id", "base", "position"]+unique_list
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in reader:
        value = row["sequence"]
        nt1 = int(row["nt1"])
        nt2 = int(row["nt2"])
        hbonds = [row["hbond1"],row["hbond2"],row["hbond3"],row["hbond4"],row["hbond5"]]

        parts = value.split("_")
        if len(parts) < 3:
            continue

        id_part = parts[0]
        sequence = parts[1]
        rest = "_".join(parts[2:])
        id_rest = f"{id_part}_{rest}"

        for i, base in enumerate(sequence, start=1):
            out_row = {col: 0 for col in fieldnames}

            out_row["id"] = id_rest
            out_row["base"] = base
            out_row["position"] = i


            for bond in hbonds:
              if bond in unique_list:
                if i == nt1:
                  out_row[bond]=max(out_row[bond],1)
                elif i == nt2:
                  out_row[bond] = max(out_row[bond],2)


            writer.writerow(out_row)

['N1(imino)*N1(imino)', 'N1(imino)*N2(amino)', 'N1(imino)*N3(imino)', 'N1(imino)*N4(amino)', 'N1(imino)-N1', 'N1(imino)-N3', 'N1(imino)-N7', 'N1(imino)-O2(carbonyl)', 'N1(imino)-O6(carbonyl)', 'N1*N1', 'N1*N3', 'N1*N7', 'N1*O2(carbonyl)', 'N1*O4(carbonyl)', 'N1-N1(imino)', 'N1-N3(imino)', 'N1-N4(amino)', 'N1-N6(amino)', 'N2(amino)*N1(imino)', 'N2(amino)*N2(amino)', 'N2(amino)-N1', 'N2(amino)-N3', 'N2(amino)-N7', 'N2(amino)-O2(carbonyl)', 'N2(amino)-O6(carbonyl)', 'N3(imino)*N1(imino)', 'N3(imino)*N3(imino)', 'N3(imino)-N1', 'N3(imino)-N3', 'N3(imino)-O2(carbonyl)', 'N3(imino)-O4(carbonyl)', 'N3(imino)-O6(carbonyl)', 'N3*N1', 'N3*N3', 'N3*O6(carbonyl)', 'N3-N1(imino)', 'N3-N2(amino)', 'N3-N3(imino)', 'N3-N4(amino)', 'N3-N6(amino)', 'N4(amino)*N1(imino)', 'N4(amino)*N2(amino)', 'N4(amino)*N3(imino)', 'N4(amino)-N1', 'N4(amino)-N3', 'N4(amino)-O2(carbonyl)', 'N4(amino)-O4(carbonyl)', 'N4(amino)-O6(carbonyl)', 'N6(amino)*N3(imino)', 'N6(amino)*N4(amino)', 'N6(amino)-N1', 'N6(amino)-N3', "N

In [ ]:
#conf_pucker

input_file=work_dir+"/good/good_clips/conf_pucker.csv"
output_file="all_conf_pucker.csv"

with open(input_file, "r") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ["id", "base","position","conf","pucker"]
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in reader:
        value = row["sequence"]
        conf=row["base_conf"]
        puck=row["puckering"]

        parts = value.split("_")
        if len(parts) < 3:
            continue

        id_part = parts[0]
        sequence = parts[1]
        rest = "_".join(parts[2:])
        id_rest = f"{id_part}_{rest}"

        for i, base in enumerate(sequence, start=1):
            out_row = {col: 0 for col in fieldnames}

            out_row["id"] = id_rest
            out_row["base"] = base
            out_row["position"] = i


            out_row["conf"] = conf
            out_row["pucker"]=puck


            writer.writerow(out_row)

In [ ]:
#bp

input_file=work_dir+"/good/good_clips/bp.csv"
output_file="all_bp.csv"

columns = ["name"]

unique_values = set()

with open(input_file, "r") as infile:
    reader = csv.DictReader(infile)

    for row in reader:
        for col in columns:
            value = (row[col] or "").strip()
            if value:  # skip empty cells
                unique_values.add(value)

# convert to list
unique_list = sorted(unique_values)
print(unique_list)
with open(input_file, "r") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.DictReader(infile)
    fieldnames = ["id", "base", "position"]+unique_list
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in reader:
        value = row["sequence"]
        nt1 = int(row["nt1"])
        nt2 = int(row["nt2"])
        bps = [row["name"]]

        parts = value.split("_")
        if len(parts) < 3:
            continue

        id_part = parts[0]
        sequence = parts[1]
        rest = "_".join(parts[2:])
        id_rest = f"{id_part}_{rest}"

        for i, base in enumerate(sequence, start=1):
            out_row = {col: 0 for col in fieldnames}

            out_row["id"] = id_rest
            out_row["base"] = base
            out_row["position"] = i


            for bp in bps:
              if bp in unique_list:
                if i == nt1:
                  out_row[bp]=max(out_row[bp],1)
                elif i == nt2:
                  out_row[bp] = max(out_row[bp],2)


            writer.writerow(out_row)

['--', 'Imino', 'Platform', 'Sheared', 'WC', 'Wobble', '~Sheared', '~Wobble', '~rWobble']


In [ ]:
import pandas as pd
import glob
from functools import reduce

path = "path/to/csv/folder"
csv_files = glob.glob(f"*.csv")

dfs = [pd.read_csv(f) for f in csv_files]

# Sequentially merge them on "id" and "position"
merged_df = reduce(lambda left, right: pd.merge(left, right, on=["id", "position","base"], how="outer"), dfs)

# Save result
merged_df.to_csv("merged_join.csv", index=False)
